# LLaMA 3.1-8B Fine-tuning on Bengali Empathetic Conversations

## Project Overview
- Fine-tune LLaMA 3.1-8B-Instruct using LoRA
- Full sequence tokenization (data-driven sequence length)
- Evaluate with Perplexity, BLEU, ROUGE
- OOP design with Strategy pattern
- Database logging for experiments

## Step 1: Install Dependencies

In [2]:
!pip install -q transformers accelerate datasets peft bitsandbytes sentencepiece
!pip install -q evaluate rouge-score nltk sacrebleu
print("Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.0 MB/s eta 0:00:00:00:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.0 MB/s eta 0:00:00
Dependencies installed


## Step 2: Imports

In [3]:
import os
import warnings
import logging
import json
import sqlite3
from datetime import datetime
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List, Dict, Tuple
import math

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)

import torch
import pandas as pd
import numpy as np
import glob
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    default_data_collator
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import login

try:
    import nltk
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except:
    pass

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

E0000 00:00:1767790095.307057      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767790095.361263      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767790095.837594      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767790095.837630      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767790095.837633      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767790095.837635      55 computation_placer.cc:177] computation placer already registered. Please check linka

PyTorch: 2.8.0+cu126
CUDA: True
GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## Step 3: Configuration

In [ ]:
@dataclass
class TrainingConfig:
    """Training configuration - optimized for T4 GPU in ~5-6 hours"""
    model_name: str = "meta-llama/Llama-3.1-8B-Instruct"
    output_dir: str = "./results"
    max_seq_length: int = 256  # Data-driven choice (see analysis below)
    batch_size: int = 4
    gradient_accumulation: int = 4
    learning_rate: float = 2e-4
    num_epochs: int = 1
    logging_steps: int = 50
    save_steps: int = 500
    warmup_ratio: float = 0.03

@dataclass  
class LoRAConfig:
    """LoRA configuration for attention layers"""
    r: int = 16
    alpha: int = 32
    dropout: float = 0.05
    target_modules: List[str] = None
    
    def __post_init__(self):
        if self.target_modules is None:
            self.target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Initialize configs
config = TrainingConfig()
lora_config = LoRAConfig()

# REPLACE WITH YOUR TOKEN!
HF_TOKEN = "Your_HF_Token"

print("Configuration:")
print(f"  Model: {config.model_name}")
print(f"  Sequence Length: {config.max_seq_length}")
print(f"  Batch Size: {config.batch_size}")
print(f"  Gradient Accumulation: {config.gradient_accumulation}")
print(f"  Effective Batch: {config.batch_size * config.gradient_accumulation}")
print(f"  LoRA Rank: {lora_config.r}")

Configuration:
  Model: meta-llama/Llama-3.1-8B-Instruct
  Sequence Length: 256
  Batch Size: 4
  Gradient Accumulation: 4
  Effective Batch: 16
  LoRA Rank: 16


## Step 4: Database Manager (Logging)

In [5]:
class DatabaseManager:
    """Manages experiment logging to SQLite database"""
    
    def __init__(self, db_path: str = "experiments.db"):
        self.db_path = db_path
        self._create_tables()
    
    def _create_tables(self):
        """Create LLAMAExperiments and GeneratedResponses tables"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS LLAMAExperiments (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                model_name TEXT NOT NULL,
                lora_config TEXT NOT NULL,
                train_loss REAL,
                val_loss REAL,
                metrics TEXT,
                timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
            )
        ''')
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS GeneratedResponses (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                experiment_id INTEGER,
                input_text TEXT NOT NULL,
                response_text TEXT NOT NULL,
                timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (experiment_id) REFERENCES LLAMAExperiments(id)
            )
        ''')
        
        conn.commit()
        conn.close()
        print("Database tables created")
    
    def log_experiment(self, model_name: str, lora_config: dict, 
                       train_loss: float, val_loss: float = None,
                       metrics: dict = None) -> int:
        """Log an experiment and return its ID"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO LLAMAExperiments (model_name, lora_config, train_loss, val_loss, metrics) VALUES (?, ?, ?, ?, ?)',
            (model_name, json.dumps(lora_config), train_loss, val_loss, json.dumps(metrics) if metrics else None)
        )
        exp_id = cursor.lastrowid
        conn.commit()
        conn.close()
        print(f"Logged experiment ID: {exp_id}")
        return exp_id
    
    def log_response(self, experiment_id: int, input_text: str, response_text: str):
        """Log a generated response"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO GeneratedResponses (experiment_id, input_text, response_text) VALUES (?, ?, ?)',
            (experiment_id, input_text, response_text)
        )
        conn.commit()
        conn.close()
    
    def get_experiments(self) -> pd.DataFrame:
        conn = sqlite3.connect(self.db_path)
        df = pd.read_sql_query("SELECT * FROM LLAMAExperiments", conn)
        conn.close()
        return df
    
    def get_responses(self, experiment_id: int = None) -> pd.DataFrame:
        conn = sqlite3.connect(self.db_path)
        query = "SELECT * FROM GeneratedResponses"
        if experiment_id:
            query += f" WHERE experiment_id = {experiment_id}"
        df = pd.read_sql_query(query, conn)
        conn.close()
        return df

db = DatabaseManager()

Database tables created


## Step 5: Dataset Processor (OOP)

In [6]:
class DatasetProcessor:
    """Handles dataset loading and preprocessing"""
    
    def __init__(self, tokenizer, max_seq_length: int):
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length
        self.raw_data = None
        self.processed_data = None
        self.dataset = None
        self.token_lengths = None
    
    def load_csv(self, file_path: str) -> pd.DataFrame:
        self.raw_data = pd.read_csv(file_path)
        print(f"Loaded {len(self.raw_data)} rows")
        return self.raw_data
    
    def auto_detect_csv(self, search_path: str = '/kaggle/input') -> str:
        csv_files = glob.glob(f'{search_path}/**/*.csv', recursive=True)
        print(f"Found: {csv_files}")
        for f in csv_files:
            if any(k in f for k in ['Bengali', 'bengali', 'Empathetic', 'Corpus']):
                return f
        return csv_files[0] if csv_files else None
    
    def analyze_token_lengths(self):
        """Analyze actual token lengths to validate sequence length choice"""
        print("Analyzing token lengths...")
        lengths = []
        for _, row in self.raw_data.iterrows():
            user = str(row["Questions"]).strip()
            assistant = str(row["Answers"]).strip()
            if user in ["", "nan"] or assistant in ["", "nan"]:
                continue
            text = f"USER: {user}\nASSISTANT: {assistant}"
            tokens = self.tokenizer(text, truncation=False)
            lengths.append(len(tokens['input_ids']))
        
        self.token_lengths = np.array(lengths)
        print(f"\nToken Length Statistics:")
        print(f"  Min: {np.min(self.token_lengths)}")
        print(f"  Max: {np.max(self.token_lengths)}")
        print(f"  Mean: {np.mean(self.token_lengths):.1f}")
        print(f"  Median: {np.median(self.token_lengths):.1f}")
        print(f"  95th percentile: {np.percentile(self.token_lengths, 95):.0f}")
        print(f"  99th percentile: {np.percentile(self.token_lengths, 99):.0f}")
        
        coverage = (self.token_lengths <= self.max_seq_length).mean() * 100
        print(f"\nMAX_SEQ_LENGTH={self.max_seq_length} covers {coverage:.1f}% of data")
        return self.token_lengths
    
    def preprocess(self, question_col: str = "Questions", answer_col: str = "Answers"):
        processed = []
        for _, row in self.raw_data.iterrows():
            user = str(row[question_col]).strip()
            assistant = str(row[answer_col]).strip()
            if user in ["", "nan"] or assistant in ["", "nan"]:
                continue
            text = f"USER: {user}\nASSISTANT: {assistant}"
            processed.append({"text": text})
        self.processed_data = processed
        print(f"Processed {len(processed)} conversation pairs")
        return processed
    
    def tokenize(self) -> Dataset:
        dataset = Dataset.from_list(self.processed_data)
        
        def tokenize_fn(examples):
            tokenized = self.tokenizer(
                examples["text"],
                truncation=True,
                padding="max_length",
                max_length=self.max_seq_length,
                return_tensors=None,
            )
            tokenized["labels"] = tokenized["input_ids"].copy()
            return tokenized
        
        self.dataset = dataset.map(tokenize_fn, batched=True, 
                                   remove_columns=dataset.column_names, desc="Tokenizing")
        print(f"Tokenized: {self.dataset}")
        return self.dataset

print("DatasetProcessor class defined")

DatasetProcessor class defined


## Step 6: Fine-tuning Strategy Pattern

In [7]:
class FineTuningStrategy(ABC):
    """Abstract base class for fine-tuning strategies"""
    
    @abstractmethod
    def prepare_model(self, model):
        pass
    
    @abstractmethod
    def get_config(self) -> dict:
        pass


class LoRAStrategy(FineTuningStrategy):
    """LoRA fine-tuning strategy"""
    
    def __init__(self, lora_cfg: LoRAConfig):
        self.lora_cfg = lora_cfg
    
    def prepare_model(self, model):
        model = prepare_model_for_kbit_training(model)
        
        peft_config = LoraConfig(
            r=self.lora_cfg.r,
            lora_alpha=self.lora_cfg.alpha,
            target_modules=self.lora_cfg.target_modules,
            lora_dropout=self.lora_cfg.dropout,
            bias="none",
            task_type="CAUSAL_LM"
        )
        
        model = get_peft_model(model, peft_config)
        model.config.use_cache = False
        
        print("\nLoRA Applied - Trainable Parameters:")
        model.print_trainable_parameters()
        return model
    
    def get_config(self) -> dict:
        return {
            "strategy": "LoRA",
            "r": self.lora_cfg.r,
            "alpha": self.lora_cfg.alpha,
            "dropout": self.lora_cfg.dropout,
            "target_modules": self.lora_cfg.target_modules
        }


class UnslothStrategy(FineTuningStrategy):
    """Unsloth fine-tuning strategy (placeholder)"""
    
    def prepare_model(self, model):
        raise NotImplementedError("Unsloth not installed. Use LoRAStrategy.")
    
    def get_config(self) -> dict:
        return {"strategy": "Unsloth"}

print("Strategy classes defined")

Strategy classes defined


## Step 7: Evaluator (OOP)

In [8]:
class Evaluator:
    """Handles model evaluation with multiple metrics"""
    
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model
        self.results = {}
    
    def calculate_perplexity(self, dataset: Dataset, max_samples: int = 100) -> float:
        self.model.eval()
        total_loss = 0
        total_tokens = 0
        samples = min(max_samples, len(dataset))
        
        with torch.no_grad():
            for i in range(samples):
                input_ids = torch.tensor([dataset[i]['input_ids']]).to(self.model.device)
                labels = torch.tensor([dataset[i]['labels']]).to(self.model.device)
                outputs = self.model(input_ids=input_ids, labels=labels)
                total_loss += outputs.loss.item() * input_ids.size(1)
                total_tokens += input_ids.size(1)
        
        perplexity = math.exp(total_loss / total_tokens)
        self.results['perplexity'] = perplexity
        print(f"Perplexity: {perplexity:.2f}")
        return perplexity
    
    def calculate_bleu(self, references: List[str], hypotheses: List[str]) -> float:
        try:
            import evaluate
            bleu = evaluate.load('sacrebleu')
            refs = [[ref] for ref in references]
            results = bleu.compute(predictions=hypotheses, references=refs)
            score = results['score']
            self.results['bleu'] = score
            print(f"BLEU: {score:.2f}")
            return score
        except Exception as e:
            print(f"BLEU error: {e}")
            return 0.0
    
    def calculate_rouge(self, references: List[str], hypotheses: List[str]) -> dict:
        try:
            import evaluate
            rouge = evaluate.load('rouge')
            results = rouge.compute(predictions=hypotheses, references=references)
            self.results['rouge1'] = results['rouge1']
            self.results['rouge2'] = results['rouge2']
            self.results['rougeL'] = results['rougeL']
            print(f"ROUGE-1: {results['rouge1']:.4f}")
            print(f"ROUGE-2: {results['rouge2']:.4f}")
            print(f"ROUGE-L: {results['rougeL']:.4f}")
            return results
        except Exception as e:
            print(f"ROUGE error: {e}")
            return {}
    
    def generate_response(self, prompt: str, max_new_tokens: int = 150) -> str:
        self.model.eval()
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
                repetition_penalty=1.1
            )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    def get_results(self) -> dict:
        return self.results

print("Evaluator class defined")

Evaluator class defined


## Step 8: LLaMA Fine-Tuner (Main OOP Class)

In [9]:
class LLAMAFineTuner:
    """Main class for LLaMA fine-tuning"""
    
    def __init__(self, config: TrainingConfig, strategy: FineTuningStrategy,
                 db_manager: DatabaseManager):
        self.config = config
        self.strategy = strategy
        self.db = db_manager
        self.tokenizer = None
        self.model = None
        self.trainer = None
        self.experiment_id = None
        print(f"LLAMAFineTuner initialized")
    
    def load_tokenizer(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "right"
        print(f"Tokenizer loaded: vocab_size={self.tokenizer.vocab_size}")
        return self.tokenizer
    
    def load_model(self):
        print("Loading model... (2-3 minutes)")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            self.config.model_name,
            quantization_config=bnb_config,
            device_map="auto",
        )
        print("Model loaded")
        return self.model
    
    def apply_strategy(self):
        self.model = self.strategy.prepare_model(self.model)
        return self.model
    
    def setup_trainer(self, train_dataset: Dataset):
        training_args = TrainingArguments(
            output_dir=self.config.output_dir,
            per_device_train_batch_size=self.config.batch_size,
            gradient_accumulation_steps=self.config.gradient_accumulation,
            learning_rate=self.config.learning_rate,
            lr_scheduler_type="cosine",
            warmup_ratio=self.config.warmup_ratio,
            num_train_epochs=self.config.num_epochs,
            fp16=True,
            bf16=False,
            gradient_checkpointing=False,  # Disabled for speed
            optim="adamw_torch",
            dataloader_num_workers=0,
            logging_steps=self.config.logging_steps,
            logging_first_step=True,
            save_steps=self.config.save_steps,
            save_total_limit=2,
            report_to="none",
            remove_unused_columns=False,
            seed=42,
        )
        self.trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            data_collator=default_data_collator,
        )
        print("Trainer initialized")
        return self.trainer
    
    def train(self) -> float:
        print("Starting training...")
        torch.cuda.empty_cache()
        result = self.trainer.train()
        train_loss = result.training_loss
        print(f"\nTraining complete. Loss: {train_loss:.4f}")
        return train_loss
    
    def save_model(self):
        save_path = f"{self.config.output_dir}/lora_adapters"
        self.model.save_pretrained(save_path)
        self.tokenizer.save_pretrained(save_path)
        print(f"Model saved to {save_path}")
        return save_path
    
    def log_experiment(self, train_loss: float, metrics: dict = None):
        self.experiment_id = self.db.log_experiment(
            model_name=self.config.model_name,
            lora_config=self.strategy.get_config(),
            train_loss=train_loss,
            metrics=metrics
        )
        return self.experiment_id
    
    def log_response(self, input_text: str, response_text: str):
        if self.experiment_id:
            self.db.log_response(self.experiment_id, input_text, response_text)

print("LLAMAFineTuner class defined")

LLAMAFineTuner class defined


## Step 9: Login and Initialize

In [10]:
login(token=HF_TOKEN)
print("Logged into HuggingFace")

Logged into HuggingFace


In [11]:
# Initialize with LoRA strategy
strategy = LoRAStrategy(lora_config)
fine_tuner = LLAMAFineTuner(config, strategy, db)
tokenizer = fine_tuner.load_tokenizer()

LLAMAFineTuner initialized


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded: vocab_size=128000


## Step 10: Load and Analyze Data

In [12]:
# Initialize processor and load data
processor = DatasetProcessor(tokenizer, config.max_seq_length)
data_path = processor.auto_detect_csv()
processor.load_csv(data_path)

# Analyze token lengths to validate sequence length choice
processor.analyze_token_lengths()

Found: ['/kaggle/input/bengali-empathetic-conversations-corpus/BengaliEmpatheticConversationsCorpus .csv']
Loaded 38233 rows
Analyzing token lengths...

Token Length Statistics:
  Min: 12
  Max: 6736
  Mean: 226.3
  Median: 158.0
  95th percentile: 443
  99th percentile: 1938

MAX_SEQ_LENGTH=256 covers 84.2% of data


array([4088, 2166, 1239, ...,  139,  116,  162])

In [13]:
# Preprocess and tokenize
processor.preprocess()
train_dataset = processor.tokenize()
print(f"\nDataset ready: {len(train_dataset)} samples")

Processed 38210 conversation pairs


Tokenizing:   0%|          | 0/38210 [00:00<?, ? examples/s]

Tokenized: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 38210
})

Dataset ready: 38210 samples


## Step 11: Load Model and Apply LoRA

In [14]:
fine_tuner.load_model()
fine_tuner.apply_strategy()

Loading model... (2-3 minutes)


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded

LoRA Applied - Trainable Parameters:
trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

## Step 12: Training Summary

In [15]:
total_samples = len(train_dataset)
effective_batch = config.batch_size * config.gradient_accumulation
total_steps = (total_samples // effective_batch) * config.num_epochs

print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"  Model:                {config.model_name}")
print(f"  Strategy:             {strategy.get_config()['strategy']}")
print(f"  Sequence Length:      {config.max_seq_length}")
print(f"  Total samples:        {total_samples:,}")
print(f"  Batch size:           {config.batch_size}")
print(f"  Gradient accum:       {config.gradient_accumulation}")
print(f"  Effective batch:      {effective_batch}")
print(f"  Total steps:          {total_steps:,}")
print(f"  LoRA rank:            {lora_config.r}")
print(f"  Gradient checkpoint:  DISABLED (for speed)")
print(f"  Mixed precision:      FP16")
print(f"  Estimated time:       ~{total_steps * 2 / 60:.0f} minutes")
print("=" * 60)

TRAINING SUMMARY
  Model:                meta-llama/Llama-3.1-8B-Instruct
  Strategy:             LoRA
  Sequence Length:      256
  Total samples:        38,210
  Batch size:           4
  Gradient accum:       4
  Effective batch:      16
  Total steps:          2,388
  LoRA rank:            16
  Gradient checkpoint:  DISABLED (for speed)
  Mixed precision:      FP16
  Estimated time:       ~80 minutes


## Step 13: Train

In [16]:
fine_tuner.setup_trainer(train_dataset)
train_loss = fine_tuner.train()

Trainer initialized
Starting training...


Step,Training Loss
1,3.716000
50,1.466900
100,0.515600
150,0.477400
200,0.459200
250,0.452500
300,0.456600
350,0.433500
400,0.441800
450,0.439200



Training complete. Loss: 0.4453


## Step 14: Evaluate

In [17]:
evaluator = Evaluator(tokenizer, fine_tuner.model)
perplexity = evaluator.calculate_perplexity(train_dataset, max_samples=50)

Perplexity: 1.76


In [18]:
# Sample responses for human evaluation
test_prompts = [
    "আমি খুব চিন্তিত বোধ করছি।",
    "আমার বন্ধুরা আমাকে বুঝতে পারে না।",
    "আমি আমার চাকরি হারিয়েছি এবং খুব দুঃখিত।",
]

print("\n" + "=" * 60)
print("SAMPLE MODEL RESPONSES (for human evaluation)")
print("=" * 60)

generated_responses = []
for prompt in test_prompts:
    formatted = f"USER: {prompt}\nASSISTANT:"
    response = evaluator.generate_response(formatted)
    generated_responses.append(response)
    print(f"\nInput: {prompt}")
    print(f"Response: {response}")
    print("-" * 60)


SAMPLE MODEL RESPONSES (for human evaluation)

Input: আমি খুব চিন্তিত বোধ করছি।
Response: USER: আমি খুব চিন্তিত বোধ করছি।
ASSISTANT: এটা সেই অনুভূতি, হয়তো পরের বার ভালো হবে
------------------------------------------------------------

Input: আমার বন্ধুরা আমাকে বুঝতে পারে না।
Response: USER: আমার বন্ধুরা আমাকে বুঝতে পারে না।
ASSISTANT: হয়তো এটি সম্ভবত দিনের উজ্জ্বল অংশে ঘটে
------------------------------------------------------------

Input: আমি আমার চাকরি হারিয়েছি এবং খুব দুঃখিত।
Response: USER: আমি আমার চাকরি হারিয়েছি এবং খুব দুঃখিত।
ASSISTANT: ওহ, আমি এটা শুনে সত্যিই দুঃখিত. অভিনন্দন! আপনি কি সম্পূর্ণভাবে হারিয়েছেন?
------------------------------------------------------------


In [19]:
# Calculate BLEU and ROUGE (using generated vs reference from dataset)
print("\nCalculating BLEU and ROUGE...")

# Get some reference responses from the dataset
references = []
for i in range(min(3, len(processor.processed_data))):
    text = processor.processed_data[i]['text']
    if 'ASSISTANT:' in text:
        ref = text.split('ASSISTANT:')[1].strip()
        references.append(ref)

if len(references) >= 3:
    # Extract just assistant part from generated
    hypotheses = []
    for resp in generated_responses:
        if 'ASSISTANT:' in resp:
            hyp = resp.split('ASSISTANT:')[1].strip()
        else:
            hyp = resp
        hypotheses.append(hyp)
    
    evaluator.calculate_bleu(references[:len(hypotheses)], hypotheses)
    evaluator.calculate_rouge(references[:len(hypotheses)], hypotheses)
else:
    print("Not enough references for BLEU/ROUGE")


Calculating BLEU and ROUGE...


BLEU: 0.00


ROUGE-1: 0.0000
ROUGE-2: 0.0000
ROUGE-L: 0.0000


## Step 15: Save and Log

In [20]:
# Save model
save_path = fine_tuner.save_model()

# Log to database
metrics = evaluator.get_results()
exp_id = fine_tuner.log_experiment(train_loss, metrics)

# Log responses
for prompt, response in zip(test_prompts, generated_responses):
    fine_tuner.log_response(prompt, response)

print(f"\nExperiment ID: {exp_id}")

Model saved to ./results/lora_adapters
Logged experiment ID: 1

Experiment ID: 1


## Step 16: Evaluation Metrics Table

In [21]:
print("\n" + "=" * 60)
print("EVALUATION METRICS TABLE")
print("=" * 60)

data_coverage = (processor.token_lengths <= config.max_seq_length).mean() * 100

metrics_table = pd.DataFrame({
    'Metric': ['Training Loss', 'Perplexity', 'BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Sequence Length', 'Data Coverage'],
    'Value': [
        f"{train_loss:.4f}",
        f"{metrics.get('perplexity', 'N/A'):.2f}" if isinstance(metrics.get('perplexity'), (int, float)) else 'N/A',
        f"{metrics.get('bleu', 'N/A'):.2f}" if isinstance(metrics.get('bleu'), (int, float)) else 'N/A',
        f"{metrics.get('rouge1', 'N/A'):.4f}" if isinstance(metrics.get('rouge1'), (int, float)) else 'N/A',
        f"{metrics.get('rouge2', 'N/A'):.4f}" if isinstance(metrics.get('rouge2'), (int, float)) else 'N/A',
        f"{metrics.get('rougeL', 'N/A'):.4f}" if isinstance(metrics.get('rougeL'), (int, float)) else 'N/A',
        f"{config.max_seq_length}",
        f"{data_coverage:.1f}%"
    ]
})
print(metrics_table.to_string(index=False))


EVALUATION METRICS TABLE
         Metric  Value
  Training Loss 0.4453
     Perplexity   1.76
           BLEU   0.00
        ROUGE-1 0.0000
        ROUGE-2 0.0000
        ROUGE-L 0.0000
Sequence Length    256
  Data Coverage  84.2%


## Step 17: View Experiment Logs

In [22]:
print("\nLogged Experiments:")
print(db.get_experiments())

print("\nLogged Responses:")
print(db.get_responses())


Logged Experiments:
   id                        model_name  \
0   1  meta-llama/Llama-3.1-8B-Instruct   

                                         lora_config  train_loss val_loss  \
0  {"strategy": "LoRA", "r": 16, "alpha": 32, "dr...    0.445258     None   

                                             metrics            timestamp  
0  {"perplexity": 1.7564778871666527, "bleu": 4.5...  2026-01-07 22:58:13  

Logged Responses:
   id  experiment_id                                input_text  \
0   1              1                 আমি খুব চিন্তিত বোধ করছি।   
1   2              1         আমার বন্ধুরা আমাকে বুঝতে পারে না।   
2   3              1  আমি আমার চাকরি হারিয়েছি এবং খুব দুঃখিত।   

                                       response_text            timestamp  
0  USER: আমি খুব চিন্তিত বোধ করছি।\nASSISTANT: এট...  2026-01-07 22:58:13  
1  USER: আমার বন্ধুরা আমাকে বুঝতে পারে না।\nASSIS...  2026-01-07 22:58:13  
2  USER: আমি আমার চাকরি হারিয়েছি এবং খুব দুঃখিত।...  2026-01-07 22:58:13

## Step 18: Download

In [23]:
import shutil

shutil.make_archive("/kaggle/working/lora_adapters", 'zip', save_path)
shutil.copy("experiments.db", "/kaggle/working/experiments.db")

print("Files ready for download:")
print("  /kaggle/working/lora_adapters.zip")
print("  /kaggle/working/experiments.db")

SameFileError: 'experiments.db' and '/kaggle/working/experiments.db' are the same file

## Summary

### Configuration Used
| Setting | Value |
|---------|-------|
| Model | LLaMA 3.1-8B-Instruct |
| Strategy | LoRA |
| Quantization | 4-bit NF4 |
| LoRA Rank | 16 |
| LoRA Alpha | 32 |
| Sequence Length | 256 (data-driven) |
| Batch Size | 4 |
| Gradient Accumulation | 4 |
| Mixed Precision | FP16 |

### OOP Classes Implemented
- **LLAMAFineTuner**: Main orchestration class
- **DatasetProcessor**: Data loading and preprocessing
- **Evaluator**: Metrics (Perplexity, BLEU, ROUGE)
- **LoRAStrategy**: Strategy pattern for LoRA
- **DatabaseManager**: Experiment logging

### Database Tables
- **LLAMAExperiments**: Training runs with configs/metrics
- **GeneratedResponses**: Model outputs for evaluation